# HW3: Regression and Classification

In this assignment you will preprocess the dataset and perform some basic regression and classification tasks. The learning outcome of this part is to know how one can pre-process a real-world dataset and perform a supervised learning task, and to understand some of the fundamental mechanisms behind these tasks.

##  Grading:

Pass/Fail.

To Pass this HW you need to provide a complete and correct solution, passing all the tests.

## OUTLINE:

Data pre-processing, regression task and classification task

1. Reading the files
2. Missing Values
3. Imputing categorical variables
4. Imputing numerical variables
5. Classification with Decision Tree, single split
6. Classification with Decision Tree, Cross validation
7. Interpretation of the results

## Important instructions:

Each function you make will be considered during the grading, so it is important to strictly follow input and output instructions stated in the skeleton code.

You must not change the names of the functions, since, if you do, the tests will fail.

Since this Homework is, in part, focused on having you implement creative solutions to impute missing data, if at any point of the homework you will use functions like fillna(), SimpleImputer(), IterativeImputer(), or packages like fancyimpute, missingpy, or similar, you will fail a test designed to spot these packages. Please, try to avoid circumventing this rule, since een if you manage to pass the homework, a similar task might be in the exam, and there you would be spotted for sure.

## Homework Scenario: Cleaning and Preparing Heart Disease Data

You have recently joined the **Data Science and Analytics Unit** at the *Global Health Institute (GHI)*, a non-profit organization focused on improving cardiovascular disease diagnosis through data-driven research.  

A junior data analyst from your team, **Franco**, sends you a message:

> “Hey, welcome to the team! We’re preparing a predictive model to help doctors identify patients at risk of heart disease using clinical data from several hospitals.  
>   
> We have two related datasets:
> - **Cleveland dataset** → this will be used for **training and validation**
> - **Hungary dataset** → this will serve as our **independent test set**
>
> Unfortunately, it looks like something went wrong during the data collection process: some values appear to have been **corrupted or lost**. Before we can train any classification model, we need to **inspect and clean the data**, handle **missing or inconsistent values**, and make sure it’s ready for modeling. I'm completely lost and I have a lot of other work, can you please help me with the cleaning and with creating some baselines classification models?”

Your task is to **analyze and clean the datasets** before **building a classifier** to predict whether a patient has heart disease.

In [2]:
# these are the libraries that you will need throughout the assignment
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from matplotlib.colors import ListedColormap

from HW import *

RSEED = 8

## *1.* Reading the files

### `Task: Read the datasets from the 'datasets' folder. Use the files called cleveland.csv and hungary.csv that you have downloaded in this archive.`

## Heart Disease Dataset — Column Descriptions

Someone has changed the names of some columns in the dataset, so make sure to use this description and refer to it for the "allowed" values.

Common sense is useful when evaluating some of the features: for example, in this dataset there is no column called weight, but, if there was one, since we are talking about humans and not ethereal beings, if a patient had a value of 0 in the weight column, this value could be due to a typo, or corrupted, and would need to be cleaned in some way.

| **Column** | **Description** |
|-------------|-----------------|
| **Age** | Age of the patient (in years). This dataset only includes adult patients. |
| **Sex** | Biological sex of the patient: `1 = male`, `0 = female`. |
| **ChestPainType** | Type of chest pain experienced: <br>• `1` = typical angina <br>• `2` = atypical angina <br>• `3` = non-anginal pain <br>• `4` = asymptomatic. |
| **RestBP** | Resting blood pressure (in mm Hg) measured on admission to the hospital. |
| **Chol** | Serum cholesterol level (in mg/dl). |
| **FBS** | Fasting blood sugar: `1` if fasting blood sugar > 120 mg/dl, otherwise `0`. |
| **RestECG** | Resting electrocardiographic results: <br>• `0` = normal <br>• `1` = ST-T wave abnormality <br>• `2` = showing probable or definite left ventricular hypertrophy. |
| **MaxHR** | Maximum heart rate achieved during the exercise test. |
| **ExAng** | Exercise-induced angina: `1` = yes, `0` = no. |
| **Oldpeak** | ST depression induced by exercise relative to rest (a measure of exercise-induced ischemia). |
| **Slope** | Slope of the peak exercise ST segment: <br>• `1` = upsloping <br>• `2` = flat <br>• `3` = downsloping. |
| **Ca** | Number of major vessels (0–3) colored by fluoroscopy (a measure of blood flow). |
| **Thal** | Thalassemia test result: <br>• `3` = normal <br>• `6` = fixed defect <br>• `7` = reversible defect. |
| **Num** | Diagnosis of heart disease (target variable): <br>`0` = no heart disease, `1–4` = presence of heart disease with increasing severity. |


In [4]:
from sklearn.impute import KNNImputer
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np

In [6]:
# From the folder 'datasets', read the files cleveland.csv and hungary.csv into the dataframes cleveland and test, respectively.

cleveland = pd.read_csv('/content/cleveland.csv')  # change this
test = pd.read_csv('/content/hungary.csv')       # change this

In [7]:
cleveland.columns

Index(['Age', 'Sex', 'ChestPainType', 'RestBP', 'Chol', 'FBS', 'RestECG',
       'MaxHR', 'ExAng', 'Oldpeak', 'Slope', 'Ca', 'Thal', 'Num'],
      dtype='object')

In [8]:
test.columns

Index(['Age', 'Sex', 'ChestPainType', 'RestBP', 'Chol', 'FBS', 'RestECG',
       'MaxHR', 'ExAng', 'Oldpeak', 'Slope', 'Ca', 'Thal', 'Num'],
      dtype='object')

In [9]:
# You can uncomment this to inspact the datasets
cleveland.head(5)

,Age,Sex,ChestPainType,RestBP,Chol,FBS,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,Thal,Num
0,53.0,1.0,3.0,130.0,246.0,1.0,2.0,173.0,0.0,0.0,1.0,3.0,3.0,0
1,54.0,1.0,4.0,110.0,206.0,0.0,2.0,108.0,1.0,0.0,2.0,1.0,3.0,3
2,222.0,1.0,4.0,125.0,249.0,1.0,2.0,144.0,1.0,1.2,2.0,1.0,3.0,1
3,58.0,1.0,4.0,100.0,234.0,0.0,0.0,156.0,0.0,0.1,1.0,1.0,7.0,2
4,51.0,0.0,4.0,130.0,305.0,0.0,0.0,142.0,1.0,1.2,2.0,0.0,7.0,2


In [40]:
(cleveland['Ca']=='?').sum()

np.int64(4)

In [41]:
(cleveland['Thal']=='?').sum()

np.int64(2)

In [10]:
test.head(5)

,Age,Sex,ChestPainType,RestBP,Chol,FBS,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,Thal,Num
0,47,0,2,140,257,0,0,135,0,1.0,1,?,?,0
1,52,1,4,112,342,0,1,96,1,1.0,2,?,?,1
2,41,0,2,125,184,0,-1,-1,0,0.0,?,?,?,0
3,58,1,4,135,222,0,0,100,0,0.0,?,?,?,0
4,54,0,2,140,309,?,1,140,0,0.0,?,?,?,0


In [11]:
# if you want to see information about the dataset, uncomment:
cleveland.describe()

,Age,Sex,ChestPainType,RestBP,Chol,FBS,RestECG,MaxHR,ExAng,Oldpeak,Slope,Num
count,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000
mean,55.052805,0.693069,3.442244,131.623762,3543.082508,0.148515,1.277228,246.953795,0.343234,1.039604,1.597360,0.937294
std,16.236574,0.599270,5.080393,17.549467,57434.537877,0.356198,5.210380,1715.122158,0.540737,1.161075,0.622104,1.228536
min,-1.000000,-1.000000,1.000000,94.000000,-234.000000,0.000000,-1.000000,-1.000000,0.000000,0.000000,0.000000,0.000000
25%,47.000000,0.000000,3.000000,120.000000,211.000000,0.000000,0.000000,132.500000,0.000000,0.000000,1.000000,0.000000
50%,56.000000,1.000000,3.000000,130.000000,240.000000,0.000000,1.000000,152.000000,0.000000,0.800000,2.000000,0.000000
75%,61.000000,1.000000,4.000000,140.000000,274.500000,0.000000,2.000000,166.000000,1.000000,1.600000,2.000000,2.000000
max,222.000000,7.000000,90.000000,200.000000,1000000.000000,1.000000,90.000000,30000.000000,5.000000,6.200000,3.000000,4.000000


In [14]:
# if you want to see information about the dataset, uncomment:
test.describe()

,Age,Sex,ChestPainType,RestBP,Oldpeak,Num
count,293.000000,293.000000,293.000000,293.000000,293.000000,293.000000
mean,48.006826,0.733788,2.986348,132.583618,0.588055,0.361775
std,11.173903,0.486937,0.965049,17.607128,0.909554,0.481336
min,1.000000,0.000000,1.000000,92.000000,0.000000,0.000000
25%,42.000000,0.000000,2.000000,120.000000,0.000000,0.000000
50%,49.000000,1.000000,3.000000,130.000000,0.000000,0.000000
75%,54.000000,1.000000,4.000000,140.000000,1.000000,1.000000
max,170.000000,4.000000,4.000000,200.000000,5.000000,1.000000


In [ ]:
cleveland.isna().sum()

In [ ]:
test.isna().sum()

In [24]:
test.shape

(293, 14)

In [ ]:
cleveland.dtypes

In [ ]:
test.dtypes

In [23]:
(test['Slope'] == '?').sum()

np.int64(188)

In [25]:
(test['Ca'] == '?').sum()

np.int64(289)

## *2.* Missing values

### `Task: use the function clean_data from the HW.py file to get a clean version of the cleveland and test dataframes.`

In [77]:
def clean_data(df):
    """
    Cleans the Cleveland heart disease dataset strictly based on the provided description:
    - Replaces '?' with NaN
    - Converts all columns to numeric dtype
    - Replaces invalid/out-of-spec values with NaN:
        * categorical columns outside allowed values
        * continuous columns with negative values
    - Returns cleaned DataFrame and missing value counts
    """

    df = df.copy()

    # Define columns
    categorical_columns = [
        'Sex', 'ChestPainType', 'FBS', 'RestECG',
        'ExAng', 'Slope', 'Ca', 'Thal'
    ]
    numerical_columns = [
        'Age', 'RestBP', 'Chol', 'MaxHR', 'Oldpeak'
    ]

    # --- Step 1: Replace '?' with NaN ---
    df = df.replace('?', np.nan)

    # --- Step 2: Convert all columns to numeric (safe conversion) ---
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # --- Step 3a: Validate categorical columns ---
    valid_values = {
        'Sex': [0, 1],
        'ChestPainType': [1, 2, 3, 4],
        'FBS': [0, 1],
        'RestECG': [0, 1, 2],
        'ExAng': [0, 1],
        'Slope': [1, 2, 3],
        'Ca': [0, 1, 2, 3],
        'Thal': [3, 6, 7],
        'Num': [0, 1, 2, 3, 4]
    }

    for col in categorical_columns + ['Num']:
        if col in df.columns:
            df.loc[~df[col].isin(valid_values[col]), col] = np.nan

    # --- Step 3b: Validate continuous numeric columns ---
    for col in numerical_columns:
        if col in df.columns:
            df.loc[df[col] < 0, col] = np.nan

    # --- Step 4: Missing value count ---
    missing_values_count = df.isna().sum().to_dict()

    return df, missing_values_count


In [78]:
# Write your code here
# cleveland_cleaned, missing_values_cleveland = pd.DataFrame(), {} # change this
# test_cleaned, missing_values_test = pd.DataFrame(), {} # change this

cleveland_cleaned, missing_values_cleveland = clean_data(cleveland) # change this
test_cleaned, missing_values_test = clean_data(test) # change this
print(missing_values_test)
print(missing_values_cleveland)

{'Age': 0, 'Sex': 1, 'ChestPainType': 0, 'RestBP': 0, 'Chol': 23, 'FBS': 8, 'RestECG': 2, 'MaxHR': 2, 'ExAng': 1, 'Oldpeak': 0, 'Slope': 188, 'Ca': 289, 'Thal': 264, 'Num': 0}
{'Age': 1, 'Sex': 2, 'ChestPainType': 1, 'RestBP': 0, 'Chol': 1, 'FBS': 0, 'RestECG': 2, 'MaxHR': 1, 'ExAng': 1, 'Oldpeak': 0, 'Slope': 1, 'Ca': 5, 'Thal': 3, 'Num': 0}


In [79]:
test_cleaned.isna().sum()

,0
Age,0
Sex,1
ChestPainType,0
RestBP,0
Chol,23
FBS,8
RestECG,2
MaxHR,2
ExAng,1
Oldpeak,0


In [80]:
cleveland_cleaned.columns

Index(['Age', 'Sex', 'ChestPainType', 'RestBP', 'Chol', 'FBS', 'RestECG',
       'MaxHR', 'ExAng', 'Oldpeak', 'Slope', 'Ca', 'Thal', 'Num'],
      dtype='object')

## *3.* Imputing categorical variables

At the beginning of this file you can find the names of the columns and a description of their contents.

Determine which columns are categorical, and set their type to object.

Determine which columns are numerical, and set their type accordingly.

Do not include the target column in any of these lists!

In [81]:
categorical_columns = [
    'Sex',           # 0 = female, 1 = male
    'ChestPainType', # 1–4
    'FBS',           # 0/1
    'RestECG',       # 0–2
    'ExAng',         # 0/1
    'Slope',         # 1–3
    'Ca',            # 0–3
    'Thal'           # 3,6,7
]      # change this
numerical_columns = [
    'Age',     # continuous
    'RestBP',  # continuous
    'Chol',    # continuous
    'MaxHR',   # continuous
    'Oldpeak'  # continuous
]
target = 'Num'

In [84]:
cleveland_cleaned['Age'].min()

4.0

### ` Task: Split the cleveland_cleaned dataframe in a train and a validation set, using train_test_split from sklearn. `

The train set must be called train, the validation set must be called val. The size of the validation set must be 30% of the total size of the cleveland_cleaned dataframe. Use shuffle=True and stratify the split based on y_cleveland. Make sure that both train and val are dataframes, and that the columns have the correct names. Reset the indexes of all four the dataframes, using drop=True.

In [85]:
# Split the data into X and y, where X contains the features and y contains the target variable.
X_cleveland = cleveland_cleaned.drop(columns=['Num'])  # change this
y_cleveland = cleveland_cleaned['Num']  # change this

X_test = test_cleaned.drop(columns=['Num'])     # change this
y_test = test_cleaned['Num']      # change this

# For test data, just reset index
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [86]:
X_train, X_val, y_train, y_val = train_test_split(
    X_cleveland,
    y_cleveland,
    test_size=0.3,
    shuffle=True,
    stratify=y_cleveland,
    random_state=42
) # change this


# Reset indexes to keep consistent DataFrames
X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)


In [87]:
print(type(X_train))

<class 'pandas.core.frame.DataFrame'>


In [62]:
# # if you want to see information about the split dataset, uncomment:
X_train.head(5)

,Age,Sex,ChestPainType,RestBP,Chol,FBS,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,Thal
210,46.0,1.0,4.0,120.0,249.0,0.0,2.0,144.0,0.0,0.8,1.0,0.0,7.0
169,52.0,1.0,2.0,128.0,205.0,1.0,0.0,184.0,0.0,0.0,1.0,0.0,3.0
75,62.0,0.0,3.0,130.0,263.0,0.0,0.0,97.0,0.0,1.2,2.0,1.0,7.0
43,58.0,1.0,3.0,140.0,211.0,1.0,2.0,165.0,0.0,0.0,1.0,0.0,3.0
201,50.0,0.0,4.0,110.0,254.0,0.0,2.0,159.0,0.0,0.0,1.0,0.0,3.0


In [63]:
# # if you want to see information about the split dataset, uncomment:
X_val.head(5)

,Age,Sex,ChestPainType,RestBP,Chol,FBS,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,Thal
199,56.0,0.0,2.0,140.0,-234.0,0.0,2.0,153.0,0.0,1.3,2.0,0.0,3.0
46,47.0,1.0,3.0,130.0,253.0,0.0,0.0,179.0,0.0,0.0,1.0,0.0,3.0
31,61.0,1.0,4.0,148.0,203.0,0.0,0.0,161.0,0.0,0.0,1.0,1.0,7.0
219,59.0,1.0,3.0,126.0,218.0,1.0,0.0,134.0,0.0,2.2,2.0,1.0,6.0
168,43.0,1.0,4.0,150.0,247.0,0.0,0.0,171.0,0.0,1.5,1.0,0.0,3.0


In [90]:
X_train.min()

,0
Age,4.0
Sex,0.0
ChestPainType,1.0
RestBP,94.0
Chol,126.0
FBS,0.0
RestECG,0.0
MaxHR,0.0
ExAng,0.0
Oldpeak,0.0


In [88]:
# To make the classification task easier, transform the target variable into a binary variable.
# If the target variable is 0, it should remain 0riable is more than 0, . If the target vait should be transformed into 1.
# Using list comprehension
y_train = pd.DataFrame([0 if i == 0 else 1 for i in y_train], columns=['Num'])
y_val   = pd.DataFrame([0 if i == 0 else 1 for i in y_val], columns=['Num'])
y_test  = pd.DataFrame([0 if i == 0 else 1 for i in y_test], columns=['Num'])

In [89]:
X_train.describe()

,Age,Sex,ChestPainType,RestBP,Chol,FBS,RestECG,MaxHR,ExAng,Oldpeak,Slope,Ca,Thal
count,212.000000,211.000000,212.000000,212.000000,212.000000,212.000000,210.000000,211.000000,211.000000,212.000000,212.000000,208.000000,209.000000
mean,54.566038,0.668246,3.198113,131.580189,4963.301887,0.183962,1.000000,289.881517,0.350711,1.040094,1.599057,0.600962,4.760766
std,13.909333,0.471963,0.927922,16.602637,68663.304324,0.388370,0.992797,2055.226274,0.478327,1.120700,0.611578,0.926997,1.946444
min,4.000000,0.000000,1.000000,94.000000,126.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,3.000000
25%,47.000000,0.000000,3.000000,120.000000,212.000000,0.000000,0.000000,132.000000,0.000000,0.000000,1.000000,0.000000,3.000000
50%,55.000000,1.000000,3.000000,130.000000,241.500000,0.000000,1.000000,152.000000,0.000000,0.800000,2.000000,0.000000,3.000000
75%,61.000000,1.000000,4.000000,140.000000,275.250000,0.000000,2.000000,167.000000,1.000000,1.650000,2.000000,1.000000,7.000000
max,197.000000,1.000000,4.000000,200.000000,1000000.000000,1.000000,2.000000,30000.000000,1.000000,6.200000,3.000000,3.000000,7.000000


### ` Task: use the impute_missing_categorical function from the HW.py file to impute the missing data from the categorical features in your dataframes. `

In [92]:
# Task 3: Categorical Features Imputation

def impute_missing_categorical(df_train, df_val, df_test, categorical_columns):
    """
    Task: Categorical Features Imputation
    --------------------------------------
    This function should handle missing values in categorical columns using appropriate techniques.
    We will skip scaling and encoding to keep things simple.

    Instructions:
    - Create subsets of the input DataFrames (train, validation, test) with only the categorical columns.
    - Use KNNImputer with k=5 and weights set to 'distance' to fill missing values in the categorical columns.
    - Ensure that the imputed values are approximated to the nearest value in the original dataset for each column, to avoid artifacts like decimal values.
    - If any "new value" is equidistant from two original values, choose the smaller one.
    - Add the column names to the resulting DataFrames after imputation.
    - The imputed dataframes should only contain the categorical columns.

    Parameters:
    df_train (pd.DataFrame): The training DataFrame.
    df_val (pd.DataFrame): The validation DataFrame.
    df_test (pd.DataFrame): The test DataFrame.
    categorical_columns (list): A list of column names corresponding to categorical features.

    Returns:
    pd.DataFrame: The training DataFrame with imputed categorical features.
    pd.DataFrame: The validation DataFrame with imputed categorical features.
    pd.DataFrame: The test DataFrame with imputed categorical features.
    """

    X_train_cat = df_train[categorical_columns].copy()
    X_val_cat = df_val[categorical_columns].copy()
    X_test_cat = df_test[categorical_columns].copy()

    imputer = KNNImputer(n_neighbors=5, weights='distance')
    imputer.fit(X_train_cat)

    # --- Step 3: Transform all sets ---
    X_train_imputed = imputer.transform(X_train_cat)
    X_val_imputed = imputer.transform(X_val_cat)
    X_test_imputed = imputer.transform(X_test_cat)

    for i, col in enumerate(categorical_columns):
        valid_values = np.sort(df_train[col].dropna().unique())  # original values
        # Function to snap each imputed value to nearest valid value
        def snap_to_nearest(value):
            diffs = np.abs(valid_values - value)
            min_diff = diffs.min()
            nearest_values = valid_values[diffs == min_diff]
            return nearest_values.min()  # pick smaller if tie
        X_train_imputed[:, i] = np.vectorize(snap_to_nearest)(X_train_imputed[:, i])
        X_val_imputed[:, i] = np.vectorize(snap_to_nearest)(X_val_imputed[:, i])
        X_test_imputed[:, i] = np.vectorize(snap_to_nearest)(X_test_imputed[:, i])

    # --- Step 5: Convert back to DataFrames with correct column names ---
    X_train_imputed = pd.DataFrame(X_train_imputed, columns=categorical_columns)
    X_val_imputed = pd.DataFrame(X_val_imputed, columns=categorical_columns)
    X_test_imputed = pd.DataFrame(X_test_imputed, columns=categorical_columns)

    return X_train_imputed, X_val_imputed, X_test_imputed



In [93]:
# Write your code here
X_train_imputed_cat, X_val_imputed_cat, X_test_imputed_cat = impute_missing_categorical(X_train, X_val, X_test, categorical_columns)

In [96]:
X_train_imputed_cat.isna().sum()

,0
Sex,0
ChestPainType,0
FBS,0
RestECG,0
ExAng,0
Slope,0
Ca,0
Thal,0


## *4.* Imputing numerical variables

` Task: use the impute_missing_numeric function from the HW.py file to impute the missing data from the numeric features in your dataframes. `

In [101]:
from sklearn.linear_model import Lasso
import pandas as pd
import numpy as np

def impute_numerical_features(df_train, df_val, df_test, numerical_columns):
    """
    Iterative Lasso imputation for numerical columns following the given steps.
    """

    # Step 1-3: Subset numerical columns
    train_num = df_train[numerical_columns].copy()
    val_num   = df_val[numerical_columns].copy()
    test_num  = df_test[numerical_columns].copy()

    # Keep original row order
    train_index = train_num.index
    val_index   = val_num.index
    test_index  = test_num.index

    # Step 4-8: Iterative imputation
    while train_num.isna().any().any():  # repeat until all missing values in train are imputed

        # Step 4a & 4b
        train_num_missing     = train_num[train_num.isna().any(axis=1)]
        train_num_not_missing = train_num.dropna(subset=[col for col in numerical_columns if col != col_to_impute]) \
                                       if 'col_to_impute' in locals() else train_num.dropna()

        # Step 5a & 5b
        val_num_missing     = val_num[val_num.isna().any(axis=1)]
        val_num_not_missing = val_num.dropna(subset=numerical_columns)

        # Step 6a & 6b
        test_num_missing     = test_num[test_num.isna().any(axis=1)]
        test_num_not_missing = test_num.dropna(subset=numerical_columns)

        # Step 7: Pick column with fewest missing values in train
        missing_counts = train_num_missing.isna().sum()
        col_to_impute = missing_counts.idxmin()

        # --- Prepare training data ---
        # Use only rows where target is not missing
        rows_with_target = train_num[train_num[col_to_impute].notna()].index
        X_full = train_num.drop(columns=[col_to_impute])
        # Keep only columns with no NaNs in these rows
        available_cols = X_full.columns[X_full.loc[rows_with_target].notna().all()]
        X_train = X_full.loc[rows_with_target, available_cols]
        y_train = train_num.loc[rows_with_target, col_to_impute]

        # If no valid features, fallback to mean
        if X_train.shape[1] == 0:
            mean_value = y_train.mean()
            train_num.loc[train_num_missing.index, col_to_impute] = mean_value
            val_num.loc[val_num_missing.index, col_to_impute] = mean_value
            test_num.loc[test_num_missing.index, col_to_impute] = mean_value
            continue

        # Step 7a: Train Lasso
        model = Lasso(alpha=0.01, max_iter=1000)
        model.fit(X_train, y_train)

        # Step 7b: Predict missing values
        if not train_num_missing.empty:
            X_missing_train = train_num_missing[available_cols]
            train_num.loc[train_num_missing.index, col_to_impute] = model.predict(X_missing_train)

        if not val_num_missing.empty:
            X_missing_val = val_num_missing[available_cols]
            val_num.loc[val_num_missing.index, col_to_impute] = model.predict(X_missing_val)

        if not test_num_missing.empty:
            X_missing_test = test_num_missing[available_cols]
            test_num.loc[test_num_missing.index, col_to_impute] = model.predict(X_missing_test)

    # Step 9-12: Save results and maintain original order
    train_num_imputed_lasso = train_num.loc[train_index]
    val_num_imputed_lasso   = val_num.loc[val_index]
    test_num_imputed_lasso  = test_num.loc[test_index]

    # Step 10-11: Final imputed datasets (numerical only)
    train_imputed_lasso = train_num_imputed_lasso
    val_imputed_lasso   = val_num_imputed_lasso
    test_imputed_lasso  = test_num_imputed_lasso

    return train_imputed_lasso, val_imputed_lasso, test_imputed_lasso


In [102]:
# Impute numerical features using iterative Lasso
X_train_imputed_num, X_val_imputed_num, X_test_imputed_num = impute_numerical_features(
    df_train=X_train,
    df_val=X_val,
    df_test=X_test,
    numerical_columns=numerical_columns
)

ValueError: Input X contains NaN.
Lasso does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

### ` Task: use the merge_imputed function from the HW.py file to merge your imputed dataframes. `

In [ ]:
# Merge the train_imputed_cat and train_imputed_num datasets. Call the resulting dataset X_train_imputed.
# Merge the val_imputed_cat and val_imputed_num datasets. Call the resulting dataset X_val_imputed.
# Merge the test_imputed_cat and test_imputed_num datasets. Call the resulting dataset X_test_imputed.

# Write your code here
X_train_imputed = pd.DataFrame()
X_val_imputed = pd.DataFrame()
X_test_imputed = pd.DataFrame()


## *5.* Classification, using a single split

### ` Use the function train_and_evaluate_single_split to produce classification results for your test set.`

In [ ]:
# The hyperparameters for the tree should be:
# criterion: ['gini', 'entropy']
# max_depth: [3, 5, 10]
# The hyperparameters for the logistic regression should be:
# penalty: ['l1', 'l2']
# C: [0.1, 10]
# solver: ['liblinear']

# For each combination of hyperparameters, train a classification pipeline using your function.


from sklearn.model_selection import ParameterGrid
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import time

hyperparameters_tree = {} # change this
hyperparameters_logreg = {} # change this
performance_df = pd.DataFrame(columns=['params', 'F1 scores'])

# create a list from the grid of hyperparameters for each model, and create the models.

start = time.time() # DO NOT CHANGE/DELETE THIS LINE

for number in range(1, 11): # change this
    # call your function here, then concat the results to performance_df
    pass # remove this line

end = time.time() # DO NOT CHANGE/DELETE THIS LINE

print('Time elapsed to run the hyperparameter tuning with a single split: ', end - start) # DO NOT CHANGE/DELETE THIS LINE

Time elapsed to run the hyperparameter tuning with a single split:  3.4809112548828125e-05


In [ ]:
# Concatenate the train and validation datasets. Call the resulting datasets X and y.
X = pd.DataFrame()  # change this
y = pd.DataFrame()  # change this

# retrain the model with the best hyperparameters on the whole training dataset.
# Remember to use the same preprocessing steps as before.

# Write your code here

## *6.* Classification with Decision Tree using Cross Validation

### ` Use the function train_and_evaluate_cross_validation to produce classification results for your test set.`

In [ ]:
# 1. Use the same hyperparameters from the previous task.
# 2. Create a dataframe to store the performance of the model with cross-validation, containing the columns 'params' and 'Average F1 scores'
# 3. You can reuse the parameter grids from the previous step.
# 4. Run your function for each combination of hyperparameters, using 5-fold cross-validation.
# 5. Concatenate the results to the dataframe created in step 2.



X = [[5,6], [10,11], [15,16], [20,21], [25,26], [30,31], [35,36], [40,41], [45,46], [50,51]]    # Delete this line
y = [0,1,0,1,0,1,0,1,0,1]                                                                       # Delete this line

X = pd.DataFrame(X)                                                                             # Delete this line
y = pd.DataFrame(y)                                                                             # Delete this line


# DO NOT FORGET TO DELETE THE PREVIOUS LINES. They are only to make the empty assignment run without errors,
# but they will destroy the data you need.

performance_df_cv = pd.DataFrame(columns=['params', 'F1 scores'])

start_CV = time.time() # DO NOT CHANGE/DELETE THIS LINE

# call your function here, then concat the results to performance_df_cv

end_CV = time.time() # DO NOT CHANGE/DELETE THIS LINE

print('Time elapsed to run the hyperparameter tuning with Cross Validation: ', end_CV - start_CV) # DO NOT CHANGE/DELETE THIS LINE


Time elapsed to run the hyperparameter tuning with Cross Validation:  0.0020859241485595703


In [ ]:
# retrain the model with the best hyperparameters on the whole training dataset.
# Remember to use the same preprocessing steps as before.

## *7.* Interpretation of the results

### ` Which model performs the best? `

Write your explanation here. Delete this text.

### ` Task: use the best model to produce predictions on the test set, then calculate the F1 score on the test set. What do you notice? `

### ` What is a possible explanation? `